# 3D Electromagnetic Modeling for Underwater Archaeological Sites

This notebook demonstrates a 3D EM (Electromagnetic) forward modeling workflow applied to the detection of archaeological structures beneath the seabed.  
The approach uses a layered-earth resistivity model combined with a 3D anomaly representing a buried structure (e.g., a shipwreck or buried walls).

**Workflow:**
1. Define the 3D model geometry (seawater layer + seabed sediments + anomaly)
2. Set up a synthetic resistivity model
3. Compute and visualise the apparent resistivity response
4. Compare responses with and without the buried anomaly

## 1. Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

# Reproducibility
np.random.seed(42)

print("Libraries loaded successfully.")

## 2. Model Grid Definition

We define a 3D Cartesian grid representing:
- **Layer 0 (z < 0 m):** Seawater — low resistivity (~0.25 Ω·m)
- **Layer 1 (0 ≤ z < 5 m):** Sandy seabed sediments — moderate resistivity (~1 Ω·m)
- **Layer 2 (5 ≤ z < 20 m):** Marine clay — low resistivity (~0.5 Ω·m)
- **Anomaly:** A resistive block buried at 6–10 m depth simulating a stone structure (e.g., ancient masonry or a wreck hull) — high resistivity (~50 Ω·m)

In [ ]:
# Grid resolution (metres)
dx, dy, dz = 2.0, 2.0, 1.0

# Grid extent
x = np.arange(-30, 31, dx)   # East-West, m
y = np.arange(-30, 31, dy)   # North-South, m
z = np.arange(-10, 21, dz)   # Depth (negative = water column, positive = below seabed)

# 3D mesh
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

print(f"Grid shape: {X.shape}  (nx={len(x)}, ny={len(y)}, nz={len(z)})")
print(f"X range: {x.min()} – {x.max()} m")
print(f"Y range: {y.min()} – {y.max()} m")
print(f"Z range: {z.min()} – {z.max()} m  (positive = depth below seabed)")

## 3. Resistivity Model Construction

In [ ]:
# Background resistivity model (layers only, Ω·m)
rho_bg = np.ones_like(X, dtype=float)

# Seawater (z < 0)
rho_bg[Z < 0]                  = 0.25
# Sandy seabed sediment (0 ≤ z < 5)
rho_bg[(Z >= 0) & (Z < 5)]     = 1.0
# Marine clay (5 ≤ z < 20)
rho_bg[(Z >= 5)]               = 0.5

# --- Anomaly model (background + buried resistive block) ---
rho_anom = rho_bg.copy()

# Resistive anomaly: stone structure buried at 6–10 m depth, 8 m × 8 m plan area
anomaly_mask = (
    (X >= -4) & (X <= 4) &
    (Y >= -4) & (Y <= 4) &
    (Z >= 6)  & (Z <= 10)
)
rho_anom[anomaly_mask] = 50.0   # High resistivity – ancient stone/masonry

print("Resistivity models built.")
print(f"  Background unique values (Ω·m): {np.unique(rho_bg)}")
print(f"  Anomaly unique values  (Ω·m): {np.unique(rho_anom)}")

## 4. Visualise the Resistivity Model — Vertical Cross-Section

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

iy = len(y) // 2  # Central Y slice (y ≈ 0 m)

for ax, rho, title in zip(axes,
                          [rho_bg, rho_anom],
                          ['Background Model', 'Anomaly Model (buried structure)']):
    im = ax.pcolormesh(
        x, z,
        np.log10(rho[:, iy, :]).T,
        cmap='RdYlBu_r',
        vmin=np.log10(0.1),
        vmax=np.log10(100)
    )
    ax.axhline(0, color='k', linewidth=1.2, linestyle='--', label='Seabed')
    ax.set_xlabel('X (m)', fontsize=12)
    ax.set_ylabel('Depth below seabed (m)', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.invert_yaxis()   # Depth increases downward
    cb = fig.colorbar(im, ax=ax, label='log₁₀ Resistivity (Ω·m)')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('resistivity_cross_section.png', dpi=150)
plt.show()
print("Figure saved as 'resistivity_cross_section.png'")

## 5. Synthetic EM Response — Apparent Resistivity Profile

We simulate a simplified **Controlled-Source EM (CSEM)** survey along a 2D profile across the buried anomaly.  
The apparent resistivity is approximated using a 1D Schlumberger-type formula applied to each model column, allowing us to compare the surface response above the background vs. the anomaly.

In [ ]:
def apparent_resistivity_1d(rho_column, dz_val, survey_depth=5.0):
    """
    Approximate apparent resistivity for a 1D layered column using
    a weighted harmonic mean (current-channeling approximation).

    Parameters
    ----------
    rho_column : 1D array  – resistivity of each cell (Ω·m)
    dz_val     : float     – cell thickness (m)
    survey_depth : float   – integration depth for the apparent resistivity (m)

    Returns
    -------
    rho_app : float – apparent resistivity (Ω·m)
    """
    n_cells = int(survey_depth / dz_val)
    rho_col = rho_column[:n_cells]
    # Harmonic mean (parallel resistors – current channeling)
    rho_app = n_cells / np.sum(1.0 / rho_col)
    return rho_app


# Compute apparent resistivity profile along X at Y=0, surface receivers (z=0)
iy      = len(y) // 2
iz_surf = np.argmin(np.abs(z - 0))   # z index closest to seabed surface

rho_app_bg   = np.zeros(len(x))
rho_app_anom = np.zeros(len(x))

for ix in range(len(x)):
    col_bg   = rho_bg  [ix, iy, iz_surf:]
    col_anom = rho_anom[ix, iy, iz_surf:]
    rho_app_bg  [ix] = apparent_resistivity_1d(col_bg,   dz, survey_depth=15.0)
    rho_app_anom[ix] = apparent_resistivity_1d(col_anom, dz, survey_depth=15.0)

print("Apparent resistivity profiles computed.")

## 6. Plot the EM Survey Response

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(x, rho_app_bg,   'b--', linewidth=2, label='Background (no anomaly)')
ax.plot(x, rho_app_anom, 'r-',  linewidth=2, label='Anomaly model (buried structure)')

ax.axvspan(-4, 4, alpha=0.12, color='orange', label='Anomaly plan extent (±4 m)')

ax.set_xlabel('X Position (m)', fontsize=12)
ax.set_ylabel('Apparent Resistivity (Ω·m)', fontsize=12)
ax.set_title('Synthetic CSEM Profile — Apparent Resistivity\n'
             'Underwater Archaeological Site (Y = 0 m)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('apparent_resistivity_profile.png', dpi=150)
plt.show()
print("Figure saved as 'apparent_resistivity_profile.png'")

## 7. 3D Visualisation of the Anomaly

In [ ]:
fig = plt.figure(figsize=(12, 7))
ax  = fig.add_subplot(111, projection='3d')

# Horizontal slice at z = 8 m (inside the anomaly)
iz_slice = np.argmin(np.abs(z - 8))
slice_rho = np.log10(rho_anom[:, :, iz_slice])

XX, YY = np.meshgrid(x, y, indexing='ij')

surf = ax.plot_surface(
    XX, YY, np.full_like(XX, z[iz_slice]),
    facecolors=cm.RdYlBu_r(
        (slice_rho - np.log10(0.1)) / (np.log10(100) - np.log10(0.1))
    ),
    alpha=0.85
)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Depth below seabed (m)')
ax.set_title('Horizontal Resistivity Slice at 8 m Depth\n'
             '(log₁₀ Ω·m — warm colours = high resistivity / anomaly)')
ax.invert_zaxis()

# Colour bar
mappable = cm.ScalarMappable(cmap='RdYlBu_r')
mappable.set_clim(np.log10(0.1), np.log10(100))
cb = fig.colorbar(mappable, ax=ax, shrink=0.5, label='log₁₀ Resistivity (Ω·m)')

plt.tight_layout()
plt.savefig('3d_resistivity_slice.png', dpi=150)
plt.show()
print("Figure saved as '3d_resistivity_slice.png'")

## 8. Anomaly Detection — Percentage Difference

Compute the relative difference between the anomaly response and the background to highlight where the buried structure produces a detectable EM signature.

In [ ]:
rel_diff_pct = 100.0 * (rho_app_anom - rho_app_bg) / rho_app_bg

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(x, rel_diff_pct, width=dx * 0.9,
       color=np.where(rel_diff_pct > 0, 'tomato', 'steelblue'),
       edgecolor='none', alpha=0.85)

ax.axhline(0, color='k', linewidth=1)
ax.axvspan(-4, 4, alpha=0.12, color='orange', label='Anomaly plan extent')
ax.set_xlabel('X Position (m)', fontsize=12)
ax.set_ylabel('Relative Difference (%)', fontsize=12)
ax.set_title('EM Anomaly Detection — Relative Difference\n'
             '(Anomaly vs Background Apparent Resistivity)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=150)
plt.show()

peak_idx = np.argmax(np.abs(rel_diff_pct))
print(f"Peak anomaly response: {rel_diff_pct[peak_idx]:.1f}% at x = {x[peak_idx]:.1f} m")

## 9. Summary

| Parameter | Value |
|---|---|
| Grid dimensions | 31 × 31 × 31 cells |
| Cell size | 2 m × 2 m × 1 m |
| Seawater resistivity | 0.25 Ω·m |
| Sandy seabed (0–5 m) | 1.0 Ω·m |
| Marine clay (5–20 m) | 0.5 Ω·m |
| Buried anomaly resistivity | 50 Ω·m |
| Anomaly depth | 6–10 m below seabed |
| Anomaly plan size | 8 m × 8 m |

The synthetic CSEM profile clearly identifies the buried resistive anomaly as a **positive apparent-resistivity anomaly** centred over the structure.  
This workflow can be extended with real inversion tools (e.g., SimPEG, MARE2DEM, or emg3d) for full 3D EM inversion.

### Next Steps
- Incorporate real bathymetric/survey geometry
- Apply 3D EM inversion using [SimPEG](https://simpeg.xyz) or [emg3d](https://emg3d.readthedocs.io)
- Integrate with sub-bottom profiler data for joint interpretation